# Notebook 5: LLM Evaluation

Reproduces the LLM-based semantic evaluation results.

**Judges:** Qwen3-4B (conservative) and Mistral-7B 4-bit (lenient)

**Systems evaluated:** BGE only, BM25 only, Full pipeline

**Metrics:** % Fully correct, % Partially correct, % Incorrect, Cohen's κ

**Dataset:** RegulationQA (n=1,002)

**Runtime:** ~60 min on T4x2

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu rank-bm25 nltk scikit-learn

import torch, json, pickle, string, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import cohen_kappa_score

In [ ]:
# Load pre-computed extraction logs
# These come from Notebook 3 or saved artifacts
print('Loading extraction logs...')

with open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/extraction_log.json') as f:
    full_pipeline_log = json.load(f)

with open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/bge_extraction_log.json') as f:
    bge_log = json.load(f)

with open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/bm25_extraction_log.json') as f:
    bm25_log = json.load(f)

print(f'Full pipeline: {len(full_pipeline_log)} items')
print(f'BGE only:      {len(bge_log)} items')
print(f'BM25 only:     {len(bm25_log)} items')

In [ ]:
# LLM judge prompt
LLM_PROMPT = """You are evaluating a regulatory information retrieval system.

QUESTION: {question}

REFERENCE ANSWER: {ground_truth}

EXTRACTED SPAN: {extracted_span}

Does the extracted span correctly answer the question?
Consider it correct even if the wording differs from the reference,
as long as the information is accurate and relevant.

Respond with exactly one of:
2 - Fully correct: span contains the complete answer
1 - Partially correct: span contains relevant but incomplete information
0 - Incorrect: span does not answer the question

Your response (single digit only):"""

def score_with_model(model, tokenizer, device, question, ground_truth, span):
    if not span or span.strip() == '':
        return 0
    prompt = LLM_PROMPT.format(
        question=question,
        ground_truth=ground_truth,
        extracted_span=span
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    import gc
    del inputs, outputs, new_tokens
    torch.cuda.empty_cache()
    gc.collect()
    for char in response:
        if char in ['0', '1', '2']:
            return int(char)
    return 0

print('Prompt and scoring function ready.')

In [ ]:
# Load Qwen3-4B on GPU 1
print('Loading Qwen3-4B...')
qwen_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')
qwen_model     = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen3-4B', torch_dtype=torch.float16, device_map='cuda:1'
)
qwen_model.eval()
print('Qwen3-4B ready on GPU 1.')

for i in range(torch.cuda.device_count()):
    free = torch.cuda.mem_get_info(i)[0] / 1024**3
    print(f'GPU {i}: {free:.1f}GB free')

In [ ]:
# Run Qwen evaluation on all three systems
def run_llm_eval(log, model, tokenizer, device, name):
    print(f'\nEvaluating {name} with Qwen3-4B...')
    scores = []
    for item in tqdm(log):
        score = score_with_model(
            model, tokenizer, device,
            item['question'], item['ground_truth'],
            item.get('best_span', '')
        )
        scores.append(score)
    scores_arr = np.array(scores)
    return {
        'fully_correct':     float((scores_arr == 2).mean()),
        'partially_correct': float((scores_arr == 1).mean()),
        'incorrect':         float((scores_arr == 0).mean()),
        'correct_partial':   float((scores_arr >= 1).mean()),
        'mean_score':        float(scores_arr.mean()),
        'scores':            scores
    }

qwen_results = {}
qwen_results['Full pipeline'] = run_llm_eval(full_pipeline_log, qwen_model, qwen_tokenizer, 'cuda:1', 'Full pipeline')
qwen_results['BGE only']      = run_llm_eval(bge_log, qwen_model, qwen_tokenizer, 'cuda:1', 'BGE only')
qwen_results['BM25 only']     = run_llm_eval(bm25_log, qwen_model, qwen_tokenizer, 'cuda:1', 'BM25 only')

print('\n=== Qwen3-4B Results ===')
for system, r in qwen_results.items():
    print(f'{system}: Correct+Partial={r["correct_partial"]*100:.1f}%  Mean={r["mean_score"]:.4f}')

In [ ]:
# Load Mistral-7B 4-bit on GPU 0
print('Loading Mistral-7B (4-bit)...')
quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'
)
mistral_tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.3')
mistral_model     = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.3', quantization_config=quant_config, device_map='cuda:0'
)
mistral_model.eval()
print('Mistral-7B ready on GPU 0.')

for i in range(torch.cuda.device_count()):
    free = torch.cuda.mem_get_info(i)[0] / 1024**3
    print(f'GPU {i}: {free:.1f}GB free')

In [ ]:
# Run Mistral evaluation on all three systems
mistral_results = {}
mistral_results['Full pipeline'] = run_llm_eval(full_pipeline_log, mistral_model, mistral_tokenizer, mistral_model.device, 'Full pipeline')
mistral_results['BGE only']      = run_llm_eval(bge_log, mistral_model, mistral_tokenizer, mistral_model.device, 'BGE only')
mistral_results['BM25 only']     = run_llm_eval(bm25_log, mistral_model, mistral_tokenizer, mistral_model.device, 'BM25 only')

print('\n=== Mistral-7B Results ===')
for system, r in mistral_results.items():
    print(f'{system}: Correct+Partial={r["correct_partial"]*100:.1f}%  Mean={r["mean_score"]:.4f}')

In [ ]:
# Cohen's kappa between Qwen and Mistral on full pipeline
kappa = cohen_kappa_score(
    qwen_results['Full pipeline']['scores'],
    mistral_results['Full pipeline']['scores']
)
if kappa >= 0.8:   interp = 'Almost perfect'
elif kappa >= 0.6: interp = 'Substantial'
elif kappa >= 0.4: interp = 'Moderate'
elif kappa >= 0.2: interp = 'Fair'
else:              interp = 'Slight'

print(f'\nCohen Kappa (Qwen vs Mistral): {kappa:.4f} ({interp})')

# Full comparison table
print('\n=== COMPLETE LLM EVALUATION TABLE ===')
print(f'{"System":<20} {"Qwen C+P":>10} {"Mistral C+P":>12}')
print('-' * 45)
for system in ['BGE only', 'BM25 only', 'Full pipeline']:
    q = qwen_results[system]["correct_partial"] * 100
    m = mistral_results[system]["correct_partial"] * 100
    print(f'{system:<20} {q:>9.1f}% {m:>11.1f}%')
print(f'\nCohen Kappa (Full pipeline): {kappa:.4f}')

In [ ]:
# Save all results
results = {
    'qwen':   {k: {m: v for m, v in r.items() if m != 'scores'} for k, r in qwen_results.items()},
    'mistral': {k: {m: v for m, v in r.items() if m != 'scores'} for k, r in mistral_results.items()},
    'cohen_kappa_full_pipeline': float(kappa),
    'kappa_interpretation': interp
}
with open('/kaggle/working/llm_evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Save combined scores for kappa
import pandas as pd
combined = pd.DataFrame({
    'question':       [item['question'] for item in full_pipeline_log],
    'ground_truth':   [item['ground_truth'] for item in full_pipeline_log],
    'best_span':      [item.get('best_span', '') for item in full_pipeline_log],
    'f1':             [item.get('f1', 0) for item in full_pipeline_log],
    'qwen_score':     qwen_results['Full pipeline']['scores'],
    'mistral_score':  mistral_results['Full pipeline']['scores'],
})
combined.to_csv('/kaggle/working/combined_llm_scores.csv', index=False)
print('Saved: llm_evaluation_results.json, combined_llm_scores.csv')